# Lab6.1 — Software and Hybrid Image Processing

**One experiment, two implementations:** complete task is Resize → Grayscale → Sobel. Route A executes entirely on the ARM; route B executes Resize in PL (using Lab5 hardware) and the rest on ARM.

**Deliverables:** correct output images; numerical comparison; mean/p95/FPS timing table for multiple sizes; engineering explanation of why a hybrid path is not a pure hardware stream. Use the same input, output geometry and algorithm for a fair comparison.

## 1. Two distinct processing paths

**A (software):** DDR → Resize (ARM) → Grayscale (ARM) → Sobel (ARM).

**B (hybrid):** DDR → DMA → Resize (PL) → DMA → DDR → Grayscale (ARM) → Sobel (ARM).

Think first: should the results be bit-for-bit identical? Differences in rounding or interpolation can be amplified by a gradient operator.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lab6_common import (load_test_image, locate_resizer, ResizerHW,
                         process_cpu, process_hybrid, measure_callable, compare_images)

image = load_test_image()
TARGET = (320, 240)  # (width, height)
print("Source:", image.shape, "Output:", TARGET, "OpenCV:", cv2.__version__)
plt.figure(figsize=(9, 5))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Shared source"); plt.axis("off"); plt.show()

## 2. Validate the two paths

The FPGA path uses **only** the known Lab5 Resize accelerator. It is not a Resize → Sobel FPGA stream. If no usable bit/hwh pair is available, the CPU experiment works and the hybrid result is honestly marked unavailable.

Compare the magnitude image and examine the absolute difference; formulate a hypothesis about Sobel sensitivity to Resize rounding.

In [ ]:
out_cpu = process_cpu(image, TARGET)
hw = None
try:
    bit = locate_resizer()
    hw = ResizerHW(bit)
    print("Hardware:", bit)
except (ImportError, FileNotFoundError, AttributeError, RuntimeError) as exc:
    print("Hybrid path unavailable:", exc)

out_hybrid = process_hybrid(image, TARGET, hw) if hw is not None else None
panels = [(out_cpu, "A / ARM")]
if out_hybrid is not None:
    metrics = compare_images(out_cpu, out_hybrid)
    diff = np.abs(out_cpu.astype(np.int16) - out_hybrid.astype(np.int16))
    print("Difference:", metrics)
    panels.extend([(out_hybrid, "B / hybrid"), (diff, "Absolute error")])
fig, axes = plt.subplots(1, len(panels), figsize=(5*len(panels), 4))
for ax, (img, label) in zip(np.atleast_1d(axes), panels):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Benchmark the **complete operation**

Use identical source data and output sizes. The hybrid time includes copy into a DMA buffer, DMA transfers, PL Resize, output copy, grayscale conversion and Sobel. The ARM time includes all software operations.

Setup of the overlay and allocation on the *first* geometry change are excluded by the warm-up; do not call these figures end-to-end video latency.

**Important:** A/B > 1 denotes a hybrid improvement; A/B < 1 denotes a slowdown. No speed-up is assumed in advance.

In [ ]:
SIZES = [(320, 240), (256, 192), (160, 120)]
REPEATS = 25
rows = []
for size in SIZES:
    a = measure_callable(lambda: process_cpu(image, size), repeats=REPEATS)
    rows.append((size, "A / ARM", a))
    if hw is not None:
        _ = process_hybrid(image, size, hw)  # allocate/configure first
        b = measure_callable(lambda: process_hybrid(image, size, hw),
                             repeats=REPEATS)
        rows.append((size, "B / hybrid", b))

print("%-12s %-12s %11s %11s %12s" %
      ("Size", "Path", "Mean ms", "P95 ms", "Processing FPS"))
for size, path, d in rows:
    print("%-12s %-12s %11.3f %11.3f %12.2f" %
          ("%dx%d" % size, path, d["mean_ms"], d["p95_ms"], d["fps_processing"]))

if hw is not None:
    for size in SIZES:
        a = next(d for s, p, d in rows if s == size and p == "A / ARM")
        b = next(d for s, p, d in rows if s == size and p == "B / hybrid")
        print("A/B speed-up for %dx%d: %.3fx" %
              (size[0], size[1], a["mean_ms"]/b["mean_ms"]))

## 4. Investigation: change the workload

Define a fourth output resolution and **predict** which route should be faster. Test your prediction with at least 20 repeated runs and state whether the result agrees with it. If the result is unexpected, discuss DMA/copy overhead and ARM OpenCV efficiency.

Distinguish three terms in the report: (1) processing-only FPS measured here, (2) application FPS including video acquisition (Lab6.3), and (3) hardware stream throughput, which requires a different FPGA design (Lab6.2).

In [ ]:
EXTRA_SIZE = (480, 320)  # Edit after writing your prediction.
a = measure_callable(lambda: process_cpu(image, EXTRA_SIZE), repeats=20)
print("ARM:", a)
if hw is not None:
    _ = process_hybrid(image, EXTRA_SIZE, hw)
    b = measure_callable(lambda: process_hybrid(image, EXTRA_SIZE, hw),
                         repeats=20)
    print("Hybrid:", b)
    print("A/B:", a["mean_ms"] / b["mean_ms"])

## 5. Report questions and cleanup

1. Where, precisely, is the resized frame written to DDR in route B?
2. Does lower ARM computation imply fewer total transfers?
3. Can measured processing-only FPS be identified with camera FPS?
4. What hardware interconnect is required to eliminate the intermediate DDR frame?

Submit figures, raw timing table, clear experimental settings and one conclusion. Stop any hardware processing before another overlay is loaded.

In [ ]:
if hw is not None:
    hw.close()
    hw = None
print("Lab6.1 complete; DMA buffers released.")